<a href="https://colab.research.google.com/github/alvaroanussa13-glitch/Projecto-de-aprendizado-de-m-quina-inibidores-InhA/blob/main/1_Preparo_do_banco_de_dados_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Desenvolvimento de modelos de machine learning baseados em QSAR-2D para a predição de novos candidatos a fármacos inibidores de Enoyl-ACP Reductase para tratamento de tuberculose***

---

*@Alvaro_Anussa*

# ***1. Preparo do banco de dados***

### **Pipeline**
1. Configuração do ambiente de trabalho
2. Selecção do dataset de compostos inibidores da TK-HER2
3. Seleção dos compostos com dados de IC₅₀
4. Limpeza dos dados (sem duplicatas, IC₅₀ e SMILES ausentes)
5. Dataset final limpo e pronto para análise

### ***1. Configuração do ambiente de trabalho***

In [ ]:
#Instalação de frameworks necessários
!pip install chembl_webresource_client #API Cliente do CHEMBL

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00


In [ ]:
# Importação de frameworks necessários
from chembl_webresource_client.new_client import new_client #Criação da API Cliente do CHEMBL
import pandas as pd # Para tabulação de dados
from tqdm import tqdm  # barra de progresso

### ***2. Selecção do dataset de compostos inibidores da TK-HER2***

In [ ]:
# Seleção do alvo InhA (Enoyl ACP - Recdutase) com base no CHEMBL ID
alvo_especifico = "CHEMBL1849"  # Enoyl-ACP Reductase
# Buscar atividades biológicas relacionadas ao alvo e os compostos
actividades = new_client.activity.filter(target_chembl_id=alvo_especifico).only(
    ["molecule_chembl_id", "standard_type", "standard_relation",
     "standard_value", "standard_units"]
)

# Criar a lista do QuerySet (permite manipular os dados requisitados)
actividades_list = list(actividades)

# Criar DataFrame com compostos
df = pd.DataFrame(actividades_list)

In [ ]:
display(df)

,molecule_chembl_id,relation,standard_relation,standard_type,standard_units,standard_value,type,units,value
0,CHEMBL217926,=,=,IC50,nM,10660.0,IC50,uM,10.66
1,CHEMBL216547,>,>,IC50,nM,100000.0,IC50,uM,100.0
2,CHEMBL213720,>,>,IC50,nM,100000.0,IC50,uM,100.0
3,CHEMBL217274,>,>,IC50,nM,100000.0,IC50,uM,100.0
4,CHEMBL217773,>,>,IC50,nM,100000.0,IC50,uM,100.0
...,...,...,...,...,...,...,...,...,...
1455,CHEMBL5631980,=,=,Inhibition,%,11.4,INH,%,11.4
1456,CHEMBL5632246,=,=,Inhibition,%,14.1,INH,%,14.1
1457,CHEMBL849,=,=,Inhibition,%,61.0,INH,%,61.0
1458,CHEMBL6149129,None,None,Inhibition,%,None,INH,None,None


In [ ]:
# Buscar informações do alvo (nome + organismo)
alvo_info = new_client.target.filter(target_chembl_id=alvo_especifico).only(
    ["pref_name", "organism"]
)
alvo_info = list(alvo_info)[0]  # pega o primeiro registro (único para este ID)

# Adicionar colunas "Alvo" e "Organismo" no DataFrame
df["Alvo"] = alvo_info["pref_name"]
df["Organismo"] = alvo_info["organism"]

# Remover duplicatas (compostos podem aparecer várias vezes em bioensaios)
df_final = df.drop_duplicates(subset=["molecule_chembl_id"]).reset_index(drop=True)

# Contar compostos únicos (percorre cada linha para contabilizar o somatório)
total_compostos_unicos = df_final.shape[0]

# Exibir resultado
display(df_final.head(10))  # mostra 10 primeiras linhas
print(f"DATASET INICIAL: {total_compostos_unicos} COMPOSTOS testados contra {alvo_info['pref_name']} ({alvo_info['organism']})")

,molecule_chembl_id,relation,standard_relation,standard_type,standard_units,standard_value,type,units,value,Alvo,Organismo
0,CHEMBL217926,=,=,IC50,nM,10660.0,IC50,uM,10.66,Enoyl-[acyl-carrier-protein] reductase [NADH],Mycobacterium tuberculosis
1,CHEMBL216547,>,>,IC50,nM,100000.0,IC50,uM,100.0,Enoyl-[acyl-carrier-protein] reductase [NADH],Mycobacterium tuberculosis
2,CHEMBL213720,>,>,IC50,nM,100000.0,IC50,uM,100.0,Enoyl-[acyl-carrier-protein] reductase [NADH],Mycobacterium tuberculosis
3,CHEMBL217274,>,>,IC50,nM,100000.0,IC50,uM,100.0,Enoyl-[acyl-carrier-protein] reductase [NADH],Mycobacterium tuberculosis
4,CHEMBL217773,>,>,IC50,nM,100000.0,IC50,uM,100.0,Enoyl-[acyl-carrier-protein] reductase [NADH],Mycobacterium tuberculosis
5,CHEMBL217273,>,>,IC50,nM,100000.0,IC50,uM,100.0,Enoyl-[acyl-carrier-protein] reductase [NADH],Mycobacterium tuberculosis
6,CHEMBL216781,>,>,IC50,nM,75000.0,IC50,uM,75.0,Enoyl-[acyl-carrier-protein] reductase [NADH],Mycobacterium tuberculosis
7,CHEMBL265016,>,>,IC50,nM,100000.0,IC50,uM,100.0,Enoyl-[acyl-carrier-protein] reductase [NADH],Mycobacterium tuberculosis
8,CHEMBL217524,=,=,IC50,nM,14830.0,IC50,uM,14.83,Enoyl-[acyl-carrier-protein] reductase [NADH],Mycobacterium tuberculosis
9,CHEMBL384149,=,=,IC50,nM,1600.0,IC50,uM,1.6,Enoyl-[acyl-carrier-protein] reductase [NADH],Mycobacterium tuberculosis


DATASET INICIAL: 973 COMPOSTOS testados contra Enoyl-[acyl-carrier-protein] reductase [NADH] (Mycobacterium tuberculosis)


### ***3. Seleção dos compostos com dados de IC₅₀***

In [ ]:
# Buscar atividades com IC50 contra HER2
atividades_ic50 = new_client.activity.filter(
    target_chembl_id=alvo_especifico,
    standard_type="IC50"
).only(["molecule_chembl_id", "standard_value"])

# Contar compostos únicos com IC50
compostos_unicos = set()
for atividade in atividades_ic50:
    if atividade.get("standard_value") is not None:
        compostos_unicos.add(atividade["molecule_chembl_id"])

In [ ]:
# Criar lista de dicionários com valores válidos de IC50
atividades_ic50_list = [
    {"molecule_chembl_id": a["molecule_chembl_id"], "IC50": a["standard_value"]}
    for a in atividades_ic50
    if a.get("standard_value") is not None
]

# Criar DataFrame
df_ic50 = pd.DataFrame(atividades_ic50_list)

# Exibir as primeiras linhas
display(df_ic50.head(10))

# Contagem de compostos únicos
print(f"DATASET SELECIONADO: {df_ic50['molecule_chembl_id'].nunique()} COMPOSTOS")

,molecule_chembl_id,IC50
0,CHEMBL217926,10660.0
1,CHEMBL216547,100000.0
2,CHEMBL213720,100000.0
3,CHEMBL217274,100000.0
4,CHEMBL217773,100000.0
5,CHEMBL217273,100000.0
6,CHEMBL216781,75000.0
7,CHEMBL265016,100000.0
8,CHEMBL217524,14830.0
9,CHEMBL384149,1600.0


DATASET SELECIONADO: 399 COMPOSTOS


### ***4. Limpeza dos dados (sem duplicatas, IC₅₀ e SMILES ausentes)***

In [ ]:
# Transformar em lista e DataFrame
atividades_list = list(atividades_ic50)
df = pd.DataFrame(atividades_list)

# Remover dados sem IC50
df = df.dropna(subset=["standard_value"])
df["IC50_nM"] = df["standard_value"].astype(float)

# Buscar os SMILES das moléculas
tqdm.pandas()

molecule = new_client.molecule
def get_smiles(chembl_id):
    try:
        mol = molecule.get(chembl_id)
        if mol.get("molecule_structures"):
            return mol["molecule_structures"]["canonical_smiles"]
    except:
        return None

# Adicionar coluna de SMILES
df["canonical_smiles"] = df["molecule_chembl_id"].progress_apply(get_smiles)

# Remover compostos sem SMILES ou IC50
df_clean = df.dropna(subset=["canonical_smiles", "IC50_nM"])

# Remover duplicatas por SMILES
df_clean = df_clean.drop_duplicates(subset="canonical_smiles")

100%|██████████| 457/457 [04:10<00:00,  1.83it/s]


### ***5. Dataset final limpo e pronto para análise***

In [ ]:
# 1. Resultado final
df_clean.head()
display(df_clean.head(10))
print(f"\nDATASET FINAL: {len(df_clean)} COMPOSTOS")

,molecule_chembl_id,standard_value,value,IC50_nM,canonical_smiles
0,CHEMBL217926,10660.0,10.66,10660.0,O=C(Nc1ccccc1)C1CC(=O)N(C2CCCCC2)C1
1,CHEMBL216547,100000.0,100.0,100000.0,O=C(Nc1ccccc1Br)C1CC(=O)N(C2CCCCC2)C1
2,CHEMBL213720,100000.0,100.0,100000.0,O=C(Nc1ccc2c(c1)OCCO2)C1CC(=O)N(C2CCCCC2)C1
3,CHEMBL217274,100000.0,100.0,100000.0,Cc1cccc(C)c1NC(=O)C1CC(=O)N(C2CCCCC2)C1
4,CHEMBL217773,100000.0,100.0,100000.0,O=C(Nc1ccc(Oc2ccccc2)cc1)C1CC(=O)N(C2CCCCC2)C1
5,CHEMBL217273,100000.0,100.0,100000.0,CCOc1ccc(NC(=O)C2CC(=O)N(C3CCCCC3)C2)cc1
6,CHEMBL216781,75000.0,75.0,75000.0,CC(=O)Nc1cccc(NC(=O)C2CC(=O)N(C3CCCCC3)C2)c1
7,CHEMBL265016,100000.0,100.0,100000.0,O=C(Nc1ccc(Cl)cc1)C1CC(=O)N(C2CCCCC2)C1
8,CHEMBL217524,14830.0,14.83,14830.0,O=C(Nc1ccc(F)c(Cl)c1)C1CC(=O)N(C2CCCCC2)C1
9,CHEMBL384149,1600.0,1.6,1600.0,COc1ccc(Cl)cc1NC(=O)C1CC(=O)N(C2CCCCC2)C1



DATASET FINAL: 399 COMPOSTOS


In [ ]:
# 2. Selecionar colunas desejadas e renomear
df_export = df_clean[["molecule_chembl_id", "canonical_smiles", "IC50_nM"]].rename(columns={
    "molecule_chembl_id": "ChEMBL_ID",
    "canonical_smiles": "SMILES",
    "IC50_nM": "IC50_nM"
})

# 3. Exportar para CSV com separador ";"
df_export.to_csv("Dataset_final.csv", sep=";", index=False)

print("Arquivo CSV salvo como 'Dataset_final.csv'")

Arquivo CSV salvo como 'Dataset_final.csv'


# **Fontes**
[API Cliente CHEMBL](https://github.com/chembl/chembl_webresource_client/blob/master/demo_wrc.ipynb)

[Documentação CHEMBL](https://chembl.gitbook.io/chembl-interface-documentation)

# ***FIM***